# Capítulo 16: Detección de Riesgos y Fraude en Tiempo Real

## Ciencia de Datos sin Filtros

En este notebook exploraremos técnicas prácticas para detectar fraude en transacciones financieras usando ciencia de datos.

**Contenido:**
1. Carga y exploración de datos
2. Detección de anomalías
3. Detección de card testing
4. Detección de account takeover
5. Detección de velocity fraud
6. Construcción del modelo de ML
7. Métricas y evaluación
8. Consideraciones éticas

> *"Los algoritmos no cometen crímenes, pero pueden señalar inocentes como culpables."*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Librerías cargadas correctamente')

## 1. Carga y Exploración de Datos

Cargaremos el dataset de transacciones y realizaremos un análisis exploratorio inicial.

In [ ]:
# Cargar datos
df = pd.read_csv('../datos/datos_transacciones_fraude.csv')

print(f'Dimensiones del dataset: {df.shape}')
print(f'\nPrimeras filas:')
df.head(10)

In [ ]:
# Información general del dataset
print('=== INFORMACIÓN DEL DATASET ===')
print(f'\nTipos de datos:')
print(df.dtypes)

print(f'\nValores nulos por columna:')
print(df.isnull().sum())

print(f'\nEstadísticas descriptivas:')
df.describe()

In [ ]:
# Distribución de fraude
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Distribución de fraude
fraud_counts = df['is_fraud'].value_counts()
axes[0].pie(fraud_counts.values, labels=['Legítimo', 'Fraude'], 
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribución de Transacciones')

# Fraude por tipo
fraud_by_type = df[df['is_fraud'] == 1]['fraud_type'].value_counts()
axes[1].barh(fraud_by_type.index, fraud_by_type.values, color=['#e74c3c', '#e67e22', '#f39c12', '#9b59b6'])
axes[1].set_title('Fraude por Tipo')
axes[1].set_xlabel('Cantidad')

# Distribución de montos
axes[2].hist(df[df['is_fraud'] == 0]['amount'], bins=50, alpha=0.7, label='Legítimo', color='#2ecc71')
axes[2].hist(df[df['is_fraud'] == 1]['amount'], bins=50, alpha=0.7, label='Fraude', color='#e74c3c')
axes[2].set_title('Distribución de Montos')
axes[2].set_xlabel('Monto ($)')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f'\nTasa de fraude: {df["is_fraud"].mean()*100:.2f}%')
print(f'Monto promedio - Legítimo: ${df[df["is_fraud"]==0]["amount"].mean():.2f}')
print(f'Monto promedio - Fraude: ${df[df["is_fraud"]==1]["amount"].mean():.2f}')

## 2. Detección de Anomalías

Las anomalías son transacciones que se desvían significativamente del comportamiento normal. Usaremos estadísticas y reglas para identificarlas.

In [ ]:
# Función para detectar anomalías basadas en estadísticas
def detect_anomalies(df):
    """
    Detecta transacciones anómalas usando múltiples criterios.
    """
    anomalies = df.copy()
    
    # 1. Anomalías por monto (Z-score)
    mean_amount = df['amount'].mean()
    std_amount = df['amount'].std()
    anomalies['amount_zscore'] = (df['amount'] - mean_amount) / std_amount
    anomalies['amount_anomaly'] = anomalies['amount_zscore'].abs() > 3
    
    # 2. Anomalías por velocity score
    anomalies['velocity_anomaly'] = df['velocity_score'] > 80
    
    # 3. Anomalías por edad de cuenta
    anomalies['new_account'] = df['account_age_days'] < 7
    
    # 4. Combinación de factores de riesgo
    anomalies['risk_score'] = (
        anomalies['amount_zscore'].abs() / 3 +
        anomalies['velocity_score'] / 100 +
        (1 / (df['account_age_days'] + 1)) * 10
    )
    
    anomalies['anomaly_flag'] = (
        anomalies['amount_anomaly'] | 
        anomalies['velocity_anomaly'] |
        anomalies['new_account']
    )
    
    return anomalies

# Aplicar detección de anomalías
df_anomalies = detect_anomalies(df)

print('=== RESULTADOS DE DETECCIÓN DE ANOMALÍAS ===')
print(f'Transacciones anómalas detectadas: {df_anomalies["anomaly_flag"].sum()}')
print(f'Porcentaje de anomalías: {df_anomalies["anomaly_flag"].mean()*100:.2f}%')

# Visualización del risk score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_anomalies[df_anomalies['is_fraud']==0]['risk_score'], 
             bins=50, alpha=0.7, label='Legítimo', color='#2ecc71')
axes[0].hist(df_anomalies[df_anomalies['is_fraud']==1]['risk_score'], 
             bins=50, alpha=0.7, label='Fraude', color='#e74c3c')
axes[0].set_title('Distribución del Risk Score')
axes[0].set_xlabel('Risk Score')
axes[0].legend()

# Scatter plot de monto vs velocity
scatter = axes[1].scatter(df_anomalies['amount'], df_anomalies['velocity_score'],
                          c=df_anomalies['is_fraud'], cmap='RdYlGn_r', alpha=0.5)
axes[1].set_title('Monto vs Velocity Score')
axes[1].set_xlabel('Monto ($)')
axes[1].set_ylabel('Velocity Score')
plt.colorbar(scatter, ax=axes[1], label='Es Fraude')

plt.tight_layout()
plt.show()

## 3. Detección de Card Testing

El **card testing** consiste en realizar múltiples transacciones de prueba con diferentes tarjetas para verificar cuáles están activas. Es como probar cada cerradura antes de encontrar la que abre.

**Patrones típicos:**
- Múltiples transacciones de monto bajo en sucesión rápida
- Diferentes tarjetas, misma dirección IP o dispositivo
- Tiempo entre transacciones menor a 60 segundos

In [ ]:
# Simular detección de card testing
def detect_card_testing(df):
    """
    Detecta patrones de card testing.
    """
    results = df.copy()
    
    # Criterios de card testing
    # 1. Transacciones de monto bajo (< $50)
    results['is_low_amount'] = df['amount'] < 50
    
    # 2. Velocity score alto (indicador de actividad rápida)
    results['high_velocity'] = df['velocity_score'] > 70
    
    # 3. Cuenta nueva
    results['new_account'] = df['account_age_days'] < 7
    
    # Score de card testing
    results['card_testing_score'] = (
        results['is_low_amount'].astype(int) * 0.3 +
        results['high_velocity'].astype(int) * 0.4 +
        results['new_account'].astype(int) * 0.3
    )
    
    results['card_testing_flag'] = results['card_testing_score'] > 0.5
    
    return results

# Aplicar detección
df_card_testing = detect_card_testing(df)

# Análisis de resultados
print('=== DETECCIÓN DE CARD TESTING ===')
card_testing_detected = df_card_testing[df_card_testing['card_testing_flag']]
print(f'Transacciones con card testing detectado: {len(card_testing_detected)}')
print(f'\nDistribución de fraudes en detecciones:')
print(card_testing_detected['is_fraud'].value_counts(normalize=True).round(3))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de montos en card testing
axes[0].hist(card_testing_detected['amount'], bins=30, color='#e74c3c', alpha=0.7)
axes[0].set_title('Distribución de Montos - Card Testing')
axes[0].set_xlabel('Monto ($)')
axes[0].axvline(x=50, color='black', linestyle='--', label='Umbral $50')
axes[0].legend()

# Card testing por tipo de fraude
ct_by_type = card_testing_detected['fraud_type'].value_counts()
axes[1].bar(ct_by_type.index, ct_by_type.values, color=['#e74c3c', '#e67e22', '#f39c12'])
axes[1].set_title('Card Testing por Tipo de Fraude')
axes[1].set_ylabel('Cantidad')

plt.tight_layout()
plt.show()

print('\n--- Insight ---')
print('Los fraudadores de card testing suelen realizar transacciones de monto bajo')
print('para "sondear" qué tarjetas están activas sin activar alertas.')

## 4. Detección de Account Takeover

El **account takeover** ocurre cuando un atacante obtiene credenciales legítimas y toma control de una cuenta. Es el equivalente digital de robar la llave maestra.

**Patrones típicos:**
- Cambio súbito de dirección IP o geolocalización
- Actualización de datos de contacto o método de pago
- Transacciones que rompen el patrón histórico del usuario

In [ ]:
# Simular detección de account takeover
def detect_account_takeover(df):
    """
    Detecta patrones de account takeover.
    """
    results = df.copy()
    
    # Criterios de account takeover
    # 1. Monto significativamente alto
    mean_amount = df['amount'].mean()
    std_amount = df['amount'].std()
    results['high_amount'] = df['amount'] > mean_amount + 2 * std_amount
    
    # 2. Cuenta estable (edad > 30 días)
    results['established_account'] = df['account_age_days'] > 30
    
    # 3. Velocity moderado (no tan alto como card testing)
    results['moderate_velocity'] = (df['velocity_score'] > 40) & (df['velocity_score'] < 80)
    
    # Score de account takeover
    results['ato_score'] = (
        results['high_amount'].astype(int) * 0.4 +
        results['established_account'].astype(int) * 0.3 +
        results['moderate_velocity'].astype(int) * 0.3
    )
    
    results['ato_flag'] = results['ato_score'] > 0.6
    
    return results

# Aplicar detección
df_ato = detect_account_takeover(df)

# Análisis de resultados
print('=== DETECCIÓN DE ACCOUNT TAKEOVER ===')
ato_detected = df_ato[df_ato['ato_flag']]
print(f'Transacciones con account takeover detectado: {len(ato_detected)}')
print(f'\nDistribución de fraudes en detecciones:')
print(ato_detected['is_fraud'].value_counts(normalize=True).round(3))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monto vs Edad de cuenta
axes[0].scatter(ato_detected['account_age_days'], ato_detected['amount'],
                c=ato_detected['is_fraud'], cmap='RdYlGn_r', alpha=0.6)
axes[0].set_title('Account Takeover: Monto vs Edad de Cuenta')
axes[0].set_xlabel('Edad de Cuenta (días)')
axes[0].set_ylabel('Monto ($)')

# Países con más account takeover
ato_countries = ato_detected['country'].value_counts().head(10)
axes[1].barh(ato_countries.index, ato_countries.values, color='#9b59b6')
axes[1].set_title('Account Takeover por País')
axes[1].set_xlabel('Cantidad')

plt.tight_layout()
plt.show()

print('\n--- Insight ---')
print('El account takeover suele afectar cuentas establecidas con montos altos.')
print('Los atacantes buscan maximizar el beneficio antes de ser detectados.')

## 5. Detección de Velocity Fraud

El **velocity fraud** explota la velocidad de procesamiento realizando muchas transacciones antes de que se active alguna alerta. Es pasar por todos los semáforos en rojo antes de que la cámara lo capture.

**Patrones típicos:**
- Alto volumen de transacciones en ventana temporal corta
- Patrones de monto repetitivos o escalonados
- Acumulación de puntos o recompensas de forma anómala

In [ ]:
# Simular detección de velocity fraud
def detect_velocity_fraud(df):
    """
    Detecta patrones de velocity fraud.
    """
    results = df.copy()
    
    # Criterios de velocity fraud
    # 1. Velocity score muy alto
    results['very_high_velocity'] = df['velocity_score'] > 85
    
    # 2. Monto en rango medio (ni muy bajo ni muy alto)
    results['medium_amount'] = (df['amount'] > 100) & (df['amount'] < 2000)
    
    # 3. Cuenta con edad intermedia
    results['intermediate_age'] = (df['account_age_days'] > 7) & (df['account_age_days'] < 60)
    
    # Score de velocity fraud
    results['velocity_score_fraud'] = (
        results['very_high_velocity'].astype(int) * 0.5 +
        results['medium_amount'].astype(int) * 0.25 +
        results['intermediate_age'].astype(int) * 0.25
    )
    
    results['velocity_flag'] = results['velocity_score_fraud'] > 0.6
    
    return results

# Aplicar detección
df_velocity = detect_velocity_fraud(df)

# Análisis de resultados
print('=== DETECCIÓN DE VELOCITY FRAUD ===')
velocity_detected = df_velocity[df_velocity['velocity_flag']]
print(f'Transacciones con velocity fraud detectado: {len(velocity_detected)}')
print(f'\nDistribución de fraudes en detecciones:')
print(velocity_detected['is_fraud'].value_counts(normalize=True).round(3))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de velocity scores
axes[0].hist(velocity_detected['velocity_score'], bins=30, color='#f39c12', alpha=0.7)
axes[0].set_title('Distribución de Velocity Score')
axes[0].set_xlabel('Velocity Score')
axes[0].axvline(x=85, color='black', linestyle='--', label='Umbral 85')
axes[0].legend()

# Velocity fraud por dispositivo
vf_by_device = velocity_detected['device_type'].value_counts()
axes[1].pie(vf_by_device.values, labels=vf_by_device.index, autopct='%1.1f%%')
axes[1].set_title('Velocity Fraud por Tipo de Dispositivo')

plt.tight_layout()
plt.show()

print('\n--- Insight ---')
print('El velocity fraud es más común en dispositivos móviles debido a la facilidad')
print('de automatizar transacciones desde aplicaciones.')

## 6. Construcción del Modelo de Machine Learning

Ahora construiremos un modelo de Machine Learning para detectar fraude de manera automatizada. Usaremos Random Forest por su robustez y interpretabilidad.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import LabelEncoder

# Preparar features para el modelo
def prepare_features(df):
    """
    Prepara las features para el modelo de ML.
    """
    features = df.copy()
    
    # Encoding de variables categóricas
    le_country = LabelEncoder()
    le_device = LabelEncoder()
    
    features['country_encoded'] = le_country.fit_transform(features['country'])
    features['device_encoded'] = le_device.fit_transform(features['device_type'])
    
    # Features derivadas
    features['log_amount'] = np.log1p(features['amount'])
    features['amount_per_day'] = features['amount'] / (features['account_age_days'] + 1)
    features['velocity_amount_ratio'] = features['velocity_score'] / (features['amount'] + 1)
    
    # Seleccionar features para el modelo
    feature_columns = [
        'amount', 'velocity_score', 'account_age_days',
        'country_encoded', 'device_encoded',
        'log_amount', 'amount_per_day', 'velocity_amount_ratio'
    ]
    
    return features[feature_columns], features['is_fraud']

# Preparar datos
X, y = prepare_features(df)

# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Datos de entrenamiento: {X_train.shape[0]} muestras')
print(f'Datos de prueba: {X_test.shape[0]} muestras')
print(f'\nDistribución de fraude en entrenamiento: {y_train.mean()*100:.2f}%')
print(f'Distribución de fraude en prueba: {y_test.mean()*100:.2f}%')

In [ ]:
# Entrenar modelo Random Forest
print('=== ENTRENAMIENTO DEL MODELO ===')
print('\nEntrenando Random Forest...')

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'  # Manejo de desbalance de clases
)

rf_model.fit(X_train, y_train)

# Predicciones
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print('\n=== EVALUACIÓN DEL MODELO ===')
print('\nReporte de clasificación:')
print(classification_report(y_test, y_pred_rf, target_names=['Legítimo', 'Fraude']))

print(f'\nAUC-ROC Score: {roc_auc_score(y_test, y_pred_proba_rf):.4f}')

## 7. Métricas y Evaluación

En detección de fraude, el **trade-off entre precisión y recall** es una cuestión ética, no solo técnica.

| Métrica | Fórmula | Significado en Fraude |
|---------|---------|----------------------|
| Precisión | TP / (TP + FP) | De los marcados como fraude, ¿cuántos realmente lo son? |
| Recall | TP / (TP + FN) | Del fraude real, ¿cuántos detectamos? |
| F1-Score | 2 × (Prec × Rec) / (Prec + Rec) | Balance entre ambos |

In [ ]:
# Visualización de métricas
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legítimo', 'Fraude'],
            yticklabels=['Legítimo', 'Fraude'])
axes[0].set_title('Matriz de Confusión')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predicción')

# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_rf)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'ROC (AUC = {roc_auc_score(y_test, y_pred_proba_rf):.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].set_title('Curva ROC')
axes[1].legend()

# Feature importance
feature_importance = pd.Series(rf_model.feature_importances_, index=X.columns)
feature_importance.sort_values(ascending=True).plot(kind='barh', ax=axes[2], color='#3498db')
axes[2].set_title('Importancia de Features')
axes[2].set_xlabel('Importancia')

plt.tight_layout()
plt.show()

# Análisis del trade-off
print('\n=== ANÁLISIS DEL TRADE-OFF PRECISIÓN-RECALL ===')
print('\nEl trade-off óptimo depende del contexto de negocio:')
print('- Si el costo de un fraude detectado es ALTO → favorecer RECALL')
print('- Si la experiencia del usuario es CRÍTICA → favorecer PRECISIÓN')
print('\nEn este caso, priorizamos recall porque es mejor prevenir que dejar pasar fraude.')

## 8. Consideraciones Éticas

> *"Un algoritmo entrenado con datos sesgados no es una herramienta justa; es un espejo de nuestras propias injusticias."*

### ¿El modelo criminaliza comunidades?

Los modelos de detección de fraude pueden perpetuar y amplificar sesgos existentes:

1. **Sesgo geográfico**: Si los datos de entrenamiento sobrerrepresentan fraudes de ciertos países, el modelo puede flaggear transacciones legítimas de esas regiones.

2. **Sesgo socioeconómico**: Usuarios de bajos ingresos que realizan transacciones inusuales (envíos familiares, pagos de emergencia) pueden ser marcados como sospechosos.

3. **Sesgo de frecuencia**: Personas que realizan muchas transacciones pequeñas (vendedores informales, trabajadores migrantes) pueden activar alertas de velocidad.

### Principios para una detección ética

1. **Transparencia**: Los usuarios deben saber que están siendo evaluados
2. **Proporcionalidad**: La respuesta debe ser proporcional al riesgo
3. **Revisión humana**: Ninguna decisión automatizada debe ser definitiva
4. **Auditoría regular**: Evaluar impactos en diferentes poblaciones
5. **Derecho a explicación**: Los usuarios afectados deben poder entender por qué

In [ ]:
# Análisis de sesgo por país
print('=== ANÁLISIS DE SESGO POR PAÍS ===')

# Calcular métricas por país
df_analysis = df.copy()
df_analysis['predicted_fraud'] = rf_model.predict(X)

# Tasa de falsos positivos por país
bias_analysis = df_analysis.groupby('country').agg(
    total=('is_fraud', 'count'),
    real_fraud=('is_fraud', 'sum'),
    predicted_fraud=('predicted_fraud', 'sum'),
    false_positives=('predicted_fraud', lambda x: (x == 1).sum() - df_analysis.loc[x.index, 'is_fraud'].sum())
).reset_index()

bias_analysis['fraud_rate'] = (bias_analysis['real_fraud'] / bias_analysis['total'] * 100).round(2)
bias_analysis['fp_rate'] = (bias_analysis['false_positives'] / bias_analysis['total'] * 100).round(2)

print('\nMétricas por país:')
print(bias_analysis[['country', 'total', 'fraud_rate', 'fp_rate']].to_string(index=False))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(bias_analysis['country'], bias_analysis['fraud_rate'], color='#e74c3c')
axes[0].set_title('Tasa de Fraude Real por País')
axes[0].set_xlabel('Tasa de Fraude (%)')

axes[1].barh(bias_analysis['country'], bias_analysis['fp_rate'], color='#f39c12')
axes[1].set_title('Tasa de Falsos Positivos por País')
axes[1].set_xlabel('Tasa de Falsos Positivos (%)')

plt.tight_layout()
plt.show()

print('\n=== RECOMENDACIONES ÉTICAS ===')
print('1. Monitorear las tasas de falsos positivos por grupo demográfico')
print('2. Implementar un sistema de apelación para usuarios bloqueados injustamente')
print('3. Realizar auditorías regulares de equidad del modelo')
print('4. Considerar umbrales diferenciados por contexto cultural y geográfico')
print('5. Documentar y comunicar las limitaciones del sistema a los usuarios')

## Resumen

En este notebook hemos explorado:

1. **Análisis exploratorio** de datos de transacciones financieras
2. **Detección de anomalías** usando estadísticas y reglas
3. **Card testing**: Múltiples transacciones de prueba con diferentes tarjetas
4. **Account takeover**: Robo de cuentas establecidas con montos altos
5. **Velocity fraud**: Explotación de la velocidad de procesamiento
6. **Modelo de Machine Learning** para automatizar la detección
7. **Métricas de evaluación** y trade-off precisión-recall
8. **Consideraciones éticas** sobre sesgo algorítmico

### Referencias

- Bolton, R. J., & Hand, D. J. (2012). Statistical fraud detection: A review. *Statistical Science*, 17(3), 235-255.
- Abdallah, A., et al. (2016). Fraud detection systems: A survey. *Journal of Network and Computer Applications*, 68, 134-145.

---

> *"La tecnología es un sirviente útil pero un amo peligroso."* — Christian Lous Lange, Nobel de la Paz 1921

*Este notebook forma parte de "Ciencia de Datos sin Filtros", una guía práctica para profesionales que buscan aplicar ciencia de datos con responsabilidad ética.*